# MSMarco Task Correctness Test

1. Query -> document mapping correctness: every query maps to the right passages, and the right passages are marked relevant by comparing parsed output against an independently reimplemented ground truth built directly from the raw dataset.

2. Score correctness: the actual production runner computes `mrr_at_10` correctly by feeding it deterministic "oracle" encoders whose retrieval behaviour is known exactly ahead of time and cross-checking the runner's output against an independent metric computation.

In [10]:
import dataloaders
import json
import math
import numpy as np
import tempfile

from dataloaders.msmarco import MSMarcoDatasetLoader
from evaluation.config import EvaluationConfig, ModelConfig, MSMarcoConfig, RunConfig
from evaluation.tasks.msmarco_retrieval import (
    MSMARCO_LANGUAGE_PAIRS,
    MSMarcoRetrievalTask,
)
from evaluation.utils.mteb_runner import run_mteb_retrieval
from mteb.models.abs_encoder import AbsEncoder
from mteb.types import PromptType
from pathlib import Path

# Repo root resolved from the installed `dataloaders` package, independent of notebook cwd.
REPO_ROOT = Path(dataloaders.__file__).resolve().parent.parent
DATA_DIR = REPO_ROOT / "data" / "ms_marco"
SPLIT = "validation"
N_QUERIES = 20

print(f"Repo root: {REPO_ROOT}")
print(f"Data dir:  {DATA_DIR}")

Repo root: /lnet/aic/personal/tomchikr/czech_embedding_benchmark
Data dir:  /lnet/aic/personal/tomchikr/czech_embedding_benchmark/data/ms_marco


## Load the data

Load 10 queries from the `validation` split via `MSMarcoDatasetLoader` (the same loader `evaluate.py` uses), and independently re-read the same 10 raw JSONL records to serve as ground truth for the correctness checks below.

In [11]:
msmarco_config = MSMarcoConfig(input_path=DATA_DIR, split=SPLIT, limit=N_QUERIES)

loader = MSMarcoDatasetLoader(dataset_dir=DATA_DIR, split=SPLIT)
dataset = loader.load(limit=N_QUERIES)
print(f"Loaded {len(dataset)} records via MSMarcoDatasetLoader")

raw_records = []
with (DATA_DIR / f"{SPLIT}.jsonl").open(encoding="utf-8") as f:
    for _ in range(N_QUERIES):
        raw_records.append(json.loads(f.readline()))
print(
    f"Loaded {len(raw_records)} records directly from disk for ground-truth comparison"
)

assert [str(r["query_id"]) for r in raw_records] == [
    str(q) for q in dataset["query_id"]
]

Loaded 20 records via MSMarcoDatasetLoader
Loaded 20 records directly from disk for ground-truth comparison


## 1. Query -> document mapping correctness

`build_expected` below is an independent reimplementation (not a reuse) of the query/corpus/relevant-docs construction that `MSMarcoRetrievalTask.load_data` performs. For each language pair, we instantiate the real task, run its real `load_data`, and assert its output is exactly equal to this independently derived ground truth.

In [12]:
LANGUAGE_FIELDS = {
    "en": {"query_field": "query", "passage_field": "passage_text"},
    "cz": {"query_field": "query_cz", "passage_field": "passage_text_cz"},
}


def build_expected(records, query_lang, passage_lang):
    """Independently derive the expected queries/corpus/relevant_docs from raw records."""
    query_field = LANGUAGE_FIELDS[query_lang]["query_field"]
    passage_field = LANGUAGE_FIELDS[passage_lang]["passage_field"]

    expected_queries = {}
    expected_corpus = {}
    expected_relevant = {}

    for record in records:
        query_id = str(record["query_id"])
        query_text = str(record[query_field]).strip()
        if not query_text:
            continue

        passages = record["passages"]
        texts = passages[passage_field]
        selected = passages["is_selected"]

        expected_queries[query_id] = query_text
        expected_relevant[query_id] = {}
        for i, text in enumerate(texts):
            doc_id = f"{query_id}:{i}"
            expected_corpus[doc_id] = {"title": "", "text": str(text).strip()}
            if selected[i]:
                expected_relevant[query_id][doc_id] = 1

    return expected_queries, expected_corpus, expected_relevant


tasks = {}
for query_lang, passage_lang in MSMARCO_LANGUAGE_PAIRS:
    task = MSMarcoRetrievalTask(
        dataset_loader=dataset,
        dataset_config=msmarco_config,
        query_lang=query_lang,
        passage_lang=passage_lang,
    )
    task.load_data()
    tasks[(query_lang, passage_lang)] = task

for (query_lang, passage_lang), task in tasks.items():
    expected_queries, expected_corpus, expected_relevant = build_expected(
        raw_records, query_lang, passage_lang
    )

    actual_queries = task.queries["test"]
    actual_corpus = task.corpus["test"]
    actual_relevant = task.relevant_docs["test"]

    assert actual_queries == expected_queries, (
        f"query mismatch for {query_lang}-{passage_lang}"
    )
    assert actual_corpus == expected_corpus, (
        f"corpus mismatch for {query_lang}-{passage_lang}"
    )
    assert actual_relevant == expected_relevant, (
        f"relevant_docs mismatch for {query_lang}-{passage_lang}"
    )

    n_judgments = sum(len(v) for v in actual_relevant.values())
    print(
        f"[{query_lang}-{passage_lang}] OK - {len(actual_queries)} queries, "
        f"{len(actual_corpus)} docs, {n_judgments} relevance judgments"
    )

[cz-cz] OK - 20 queries, 200 docs, 10 relevance judgments
[cz-en] OK - 20 queries, 200 docs, 10 relevance judgments
[en-cz] OK - 20 queries, 200 docs, 10 relevance judgments
[en-en] OK - 20 queries, 200 docs, 10 relevance judgments


### Metadata sanity checks

Every task should expose the correct task name, main score, and evaluation languages for its language pair.

In [13]:
LANG_TO_EVAL_LANG = {"cz": "ces-Latn", "en": "eng-Latn"}

for (query_lang, passage_lang), task in tasks.items():
    meta = task.metadata
    expected_name = f"msmarco_{query_lang}-{passage_lang}_retrieval"
    expected_eval_langs = sorted(
        {LANG_TO_EVAL_LANG[query_lang], LANG_TO_EVAL_LANG[passage_lang]}
    )

    assert meta.name == expected_name, (meta.name, expected_name)
    assert meta.main_score == "mrr_at_10", meta.main_score
    assert meta.eval_langs == expected_eval_langs, (
        meta.eval_langs,
        expected_eval_langs,
    )
    assert meta.prompt == {
        "query": "Given a web search query, retrieve relevant passages that answer the query"
    }

print("Task metadata OK for all language pairs")

Task metadata OK for all language pairs


## 2. Score correctness (using the current production runner)

`OracleEncoder` returns embeddings that are looked up **by document/query id** (not by text), precomputed by us ahead of time. Because we control the embeddings exactly, we know exactly what the ranking - and therefore the metrics - should be. We then run the real `run_mteb_retrieval` (the exact function `evaluation/evaluate.py` uses) and check its output against:

- a **perfect oracle**, where each relevant passage gets the exact same embedding as its query (guaranteeing a perfect ranking, so `mrr_at_10` should be exactly `1.0`);
- a **random oracle**, with no relevance signal at all, where the ranking is effectively arbitrary but fully deterministic - so we can independently recompute the exact `mrr_at_10` value from scratch and check it matches the runner's output bit-for-bit. This isolates the correctness of the *scoring arithmetic* from retrieval quality.

Since relevance judgments and document/query ids are identical across all 4 language pairs (only the passage/query *text* differs), both oracles should also produce identical scores across all 4 language pairs.

In [14]:
from mteb.models.model_meta import ModelMeta, ScoringFunction


class OracleEncoder(AbsEncoder):
    """Deterministic encoder returning precomputed embeddings, looked up by id.

    MTEB's dataloaders keep the original query/document `id` alongside the
    `text` in every batch (used internally to map embeddings back to ids after
    encoding), so we can look up the exact, known-in-advance embedding for each
    query/document id directly - no reliance on batch ordering.
    """

    def __init__(self, query_vectors, corpus_vectors, name):
        self.query_vectors = query_vectors
        self.corpus_vectors = corpus_vectors
        self.mteb_model_meta = ModelMeta(
            loader=None,
            name=name,
            revision="1",
            release_date=None,
            languages=None,
            n_parameters=None,
            memory_usage_mb=None,
            max_tokens=None,
            embed_dim=None,
            license=None,
            open_weights=True,
            public_training_code=None,
            public_training_data=None,
            framework=[],
            similarity_fn_name=ScoringFunction.COSINE,
            use_instructions=False,
            training_datasets=None,
        )

    def encode(self, inputs, *, task_metadata=None, prompt_type=None, **kwargs):
        vectors = (
            self.query_vectors
            if prompt_type == PromptType.query
            else self.corpus_vectors
        )
        embeddings = [vectors[doc_id] for batch in inputs for doc_id in batch["id"]]
        return np.asarray(embeddings, dtype=np.float32)

In [15]:
# Query/corpus ids and relevance judgments are identical across language pairs, so any
# task can serve as the reference for building oracle vectors.
reference_task = tasks[("cz", "cz")]
query_ids = list(reference_task.queries["test"].keys())
corpus_ids = list(reference_task.corpus["test"].keys())
relevant_docs = reference_task.relevant_docs["test"]

for other_task in tasks.values():
    assert list(other_task.queries["test"].keys()) == query_ids
    assert list(other_task.corpus["test"].keys()) == corpus_ids
    assert other_task.relevant_docs["test"] == relevant_docs

queries_with_positives = [qid for qid in query_ids if relevant_docs.get(qid)]
print(
    f"{len(queries_with_positives)}/{len(query_ids)} queries have at least one relevant "
    "document (MTEB filters out the rest before scoring)"
)

DIM = 32


def make_perfect_vectors(seed=0):
    """Every relevant passage gets its query's exact embedding (similarity == 1.0)."""
    rng = np.random.default_rng(seed)
    query_vectors = {qid: rng.normal(size=DIM) for qid in query_ids}
    corpus_vectors = {}
    for doc_id in corpus_ids:
        qid = doc_id.split(":")[0]
        if doc_id in relevant_docs.get(qid, {}):
            corpus_vectors[doc_id] = query_vectors[qid]
        else:
            corpus_vectors[doc_id] = rng.normal(size=DIM)
    return query_vectors, corpus_vectors


def make_random_vectors(seed=58):
    """No relevance signal at all - purely random, deterministic embeddings."""
    rng = np.random.default_rng(seed)
    query_vectors = {qid: rng.normal(size=DIM) for qid in query_ids}
    corpus_vectors = {doc_id: rng.normal(size=DIM) for doc_id in corpus_ids}
    return query_vectors, corpus_vectors


def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def manual_mrr_at_10(query_vectors, corpus_vectors):
    """From-scratch MRR@10, independent of MTEB's own implementation."""
    reciprocal_ranks = []
    for qid in queries_with_positives:
        sims = [
            (doc_id, cosine_sim(query_vectors[qid], corpus_vectors[doc_id]))
            for doc_id in corpus_ids
        ]
        sims.sort(key=lambda item: item[1], reverse=True)
        rr = 0.0
        for rank, (doc_id, _) in enumerate(sims[:10]):
            if doc_id in relevant_docs[qid]:
                rr = 1.0 / (rank + 1)
                break
        reciprocal_ranks.append(rr)
    return sum(reciprocal_ranks) / len(reciprocal_ranks)

9/20 queries have at least one relevant document (MTEB filters out the rest before scoring)


In [16]:
def run_oracle(query_vectors, corpus_vectors, label):
    """Evaluate an oracle encoder on all 4 tasks using the real production runner."""
    model_name = f"test/oracle-{label}"
    encoder = OracleEncoder(query_vectors, corpus_vectors, name=model_name)
    with tempfile.TemporaryDirectory() as tmp_dir:
        config = EvaluationConfig(
            run=RunConfig(
                output_dir=Path(tmp_dir),
                seed=42,
                batch_size=32,
                num_proc=1,
                device="cpu",
                resume_from_partial=False,
                write_predictions=False,
            ),
            msmarco=None,
            ctdc_synthetic=None,
            multilingual_mteb=None,
            models=(
                ModelConfig(
                    name=model_name,
                    normalize_embeddings=False,
                    use_safetensors=False,
                    trust_remote_code=False,
                ),
            ),
            config_path=Path("oracle-test.toml"),
            repo_root=REPO_ROOT,
        )
        results = run_mteb_retrieval(
            config=config,
            tasks=list(tasks.values()),
            models=[encoder],
            dataset_name=f"oracle_{label}",
        )
    return results[model_name]


def get_mrr_at_10(model_result, task_name):
    for task_result in model_result.task_results:
        if task_result.task_name == task_name:
            scores = task_result.scores["test"]
            assert len(scores) == 1, scores
            return scores[0]["mrr_at_10"]
    raise KeyError(task_name)

### Perfect oracle: `mrr_at_10` should be exactly `1.0` for every language pair

In [17]:
perfect_query_vectors, perfect_corpus_vectors = make_perfect_vectors()
perfect_results = run_oracle(perfect_query_vectors, perfect_corpus_vectors, "perfect")
expected_perfect_mrr = manual_mrr_at_10(perfect_query_vectors, perfect_corpus_vectors)
print(f"Manually computed MRR@10 for the perfect oracle: {expected_perfect_mrr:.6f}")

for query_lang, passage_lang in MSMARCO_LANGUAGE_PAIRS:
    task_name = f"msmarco_{query_lang}-{passage_lang}_retrieval"
    mrr_at_10 = get_mrr_at_10(perfect_results, task_name)
    print(f"[{query_lang}-{passage_lang}] runner mrr_at_10 = {mrr_at_10:.6f}")
    assert math.isclose(mrr_at_10, 1.0, abs_tol=1e-9), task_name
    assert math.isclose(mrr_at_10, expected_perfect_mrr, abs_tol=1e-9), task_name

print(
    "Perfect oracle: runner scores match the manual computation and are perfect for every language pair."
)

Evaluating tasks:   0%|          | 0/4 [00:00<?, ?it/s]

Flattening the indices:   0%|          | 0/9 [00:00<?, ? examples/s]

Standardizing text corpus format:   0%|          | 0/200 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/9 [00:00<?, ? examples/s]

Standardizing text corpus format:   0%|          | 0/200 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/9 [00:00<?, ? examples/s]

Standardizing text corpus format:   0%|          | 0/200 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/9 [00:00<?, ? examples/s]

Standardizing text corpus format:   0%|          | 0/200 [00:00<?, ? examples/s]

Manually computed MRR@10 for the perfect oracle: 1.000000
[cz-cz] runner mrr_at_10 = 1.000000
[cz-en] runner mrr_at_10 = 1.000000
[en-cz] runner mrr_at_10 = 1.000000
[en-en] runner mrr_at_10 = 1.000000
Perfect oracle: runner scores match the manual computation and are perfect for every language pair.


### Random (no-signal) oracle: runner output must match the independent manual computation exactly

This is the more rigorous check: with no relevance signal at all, `mrr_at_10` should land at some non-trivial, deterministic value that depends purely on scoring arithmetic - not on retrieval quality. If the runner's scoring pipeline had a bug (wrong rank formula, wrong query filtering, wrong corpus, ...), this value would very likely disagree with our from-scratch computation.

In [18]:
random_query_vectors, random_corpus_vectors = make_random_vectors()
random_results = run_oracle(random_query_vectors, random_corpus_vectors, "random")
expected_random_mrr = manual_mrr_at_10(random_query_vectors, random_corpus_vectors)
print(f"Manually computed MRR@10 for the random oracle: {expected_random_mrr:.6f}")

runner_scores = {}
for query_lang, passage_lang in MSMARCO_LANGUAGE_PAIRS:
    task_name = f"msmarco_{query_lang}-{passage_lang}_retrieval"
    mrr_at_10 = get_mrr_at_10(random_results, task_name)
    runner_scores[(query_lang, passage_lang)] = mrr_at_10
    print(f"[{query_lang}-{passage_lang}] runner mrr_at_10 = {mrr_at_10:.6f}")
    assert math.isclose(mrr_at_10, expected_random_mrr, abs_tol=1e-9), task_name

# Scoring depends only on ids/embeddings, not on the passage/query text - so every
# language pair must produce exactly the same score.
assert len(set(runner_scores.values())) == 1, runner_scores

print(
    "Random oracle: runner scores match the manual computation exactly, identically across all language pairs."
)

Evaluating tasks:   0%|          | 0/4 [00:00<?, ?it/s]

Flattening the indices:   0%|          | 0/9 [00:00<?, ? examples/s]

Standardizing text corpus format:   0%|          | 0/200 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/9 [00:00<?, ? examples/s]

Standardizing text corpus format:   0%|          | 0/200 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/9 [00:00<?, ? examples/s]

Standardizing text corpus format:   0%|          | 0/200 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/9 [00:00<?, ? examples/s]

Standardizing text corpus format:   0%|          | 0/200 [00:00<?, ? examples/s]

Manually computed MRR@10 for the random oracle: 0.018519
[cz-cz] runner mrr_at_10 = 0.018519
[cz-en] runner mrr_at_10 = 0.018519
[en-cz] runner mrr_at_10 = 0.018519
[en-en] runner mrr_at_10 = 0.018519
Random oracle: runner scores match the manual computation exactly, identically across all language pairs.


## Summary

For all 4 MSMarco language pairs (`cz-cz`, `cz-en`, `en-cz`, `en-en`), using the `validation` split limited to 10 queries and the actual `run_mteb_retrieval` production runner:

- Every query is mapped to exactly the right passages, and exactly the right passages are marked relevant (verified against an independently reimplemented ground truth).
- Task metadata (name, main score, eval languages, prompt) is correct for each language pair.
- `mrr_at_10` computed by the runner is exactly `1.0` for a provably perfect retriever.
- `mrr_at_10` computed by the runner exactly matches an independent, from-scratch manual computation for a retriever with no relevance signal at all, and is identical across all 4 language pairs.